In [ ]:
import pandas as pd
import requests
from datetime import datetime , timedelta
import os
from mecoda_minka import get_obs, get_dfs
import folium
from folium.plugins import HeatMap
from html2image import Html2Image
import calendar

API_PATH = "https://api.minka-sdg.org/v1"

os.makedirs("data", exist_ok=True)


# 1. Construir un dataframe de métricas principales del proyecto y variación en el último mes.

El dataframe resultante tendrá esta forma:

```python
metric, number_today, number_in_last_month
observations, 1521, 23
observers, 124, 2
identifiers, 26, 0
species, 462, 5
```

* number_today = dato actual de observaciones, observadores, identificadores, especies.
* number_in_last_month = dato de fecha actual menos dato registrado 30 días antes (variación en los últimos 30 días).

Usamos llamadas a la API, porque son datos totales. 

Creamos un directorio "data" donde guardaremos todos los csv que vamos a generar. Este dataframe lo guardamos como "data/main_metrics.cvs", sin incluir los índices (index=False).


In [ ]:
def get_main_metrics(id_project):

    """
    Obtiene las métricas principales de un proyecto y el aumento en el mes anterior.
    :param id_project: ID del proyecto
    :return: DataFrame con las métricas principales
    """

    session = requests.Session()

    # Primero he definido las fechas que se utilizaran para obtener las metricas : Defino el mes actual y el mes anterior

    # MES ACTUAL
    today = datetime.today()
   
    # MES ANTERIOR
    last_month_date = today - timedelta(days=30)
  
    # METRICAS PARA EL MES ACTUAL 

    url_observations_today = f'{API_PATH}/observations?project_id={id_project}'
    observations_results_today = session.get(url_observations_today).json()['total_results']

    url_observers_today = f'{API_PATH}/observations/observers?project_id={id_project}'
    observers_results_today = session.get(url_observers_today).json()['total_results']

    url_identifiers_today = f'{API_PATH}/observations/identifiers?project_id={id_project}'
    identifiers_results_today = session.get(url_identifiers_today).json()['total_results']

    url_species_today = f'{API_PATH}/observations/species_counts?project_id={id_project}'
    species_results_today = session.get(url_species_today).json()['total_results']

    # METRICAS PARA EL MES ANTERIOR

    url_observations_last = url_observations_today + f'&d1={last_month_date}'
    observations_results_last = session.get(url_observations_last).json()['total_results']

    url_observers_last = url_observers_today + f'&d1={last_month_date}'
    observers_results_last = session.get(url_observers_last).json()['total_results']

    url_identifiers_last = url_identifiers_today + f'&d1={last_month_date}'
    identifiers_results_last = session.get(url_identifiers_last).json()['total_results']

    url_species_last = url_species_today + f'&d1={last_month_date}'
    species_results_last = session.get(url_species_last).json()['total_results']

    # Aqui genero el dataframe para almacenar los datos (TODOS LOS DATOS SON CON EL RESEARCH GRADE)

    df_main_metrics = pd.DataFrame({
        'metric': ['observations', 'observers', 'identifiers', 'species'],
        'number_today': [observations_results_today, observers_results_today, identifiers_results_today, species_results_today],
        'number_in_last_month': [observations_results_last, observers_results_last, identifiers_results_last, species_results_last]
    })

    return df_main_metrics

In [5]:
df_main_metrics

,metric,number_today,number_in_last_month
0,observations,5654,228
1,observers,153,12
2,identifiers,68,12
3,species,653,97


# 2. Evolución de las métricas principales

Construir un dataframe con esta forma:

```python
month, observations, observers, identifiers, species
2024-01, 185, 15, 7, 62
2024-02, 128, 3, 1, 32
...
```

Los datos no son acumulativos, son del mes en concreto. Lo sacaremos usando llamadas a la API.

Para ello puedes utilizar estas funciones, que te ayudarán a construirlo:

Esto nos devuelve un diccionario con los meses como clave y el último día del mes como valor. Así podemos usarlo con la función anterior:

Ahora hay que unir las dos funciones para sacar cada mes y de cada mes sacar los valores de las métricas. Eso nos da los resultados de un mes, que podemos guardar en un diccionario. Y luego unimos los diccionarios de cada mes en una lista de todos los meses. Te pongo debajo un ejemplo de uso con un mes.

Ahora hay que crear una función que itere por todos los elementos de la lista meses desde el inicio de MINKA y saque los datos para cada mes usando get_totals a un diccionario, los acumule en la lista y la lista la convierta a un dataframe.

Es decir, para cada elemento de los meses, usamos get_totals para sacar las métricas y las almacenamos. Si lo ves complicado lo hacemos juntos.

Ese dataframe lo guardamos como "data/monthly_metrics.csv".

In [10]:
# En esta casilla se genera la funcion que recorre los meses y obtiene las metricas de cada uno de ellos
def get_totals(project_id, year, month, kind="project", session=None):
    if session is None:
        session = requests.Session()

    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"

    total_obs = session.get(url_obs).json().get("total_results", 0)
    total_part = session.get(url_part).json().get("total_results", 0)
    total_ident = session.get(url_ident).json().get("total_results", 0)
    total_spe = session.get(url_spe).json().get("total_results", 0)

    return total_obs, total_part, total_ident, total_spe


def get_month_list(years: list) -> list:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
    return meses

def build_monthly_metrics(project_id):
    meses = get_month_list(range(2022, datetime.now().year + 1))
    total_metrics = []

    session = requests.Session()

    for mes in meses:
        year = int(mes.split("-")[0])
        month = int(mes.split("-")[1])

        print(f"Procesando {mes}...")

        try:
            total_obs, total_part, total_ident, total_spe = get_totals(
                project_id=project_id, year=year, month=month, kind="project", session=session
            )
        except Exception as e:
            print(f"Error procesando {mes}: {e}")
            continue


        # Generamos el diccionario para posteriormente añadirlos a la lista total_metrics 
        total_for_month = {
            "month": mes,
            "observations": total_obs,
            "observers": total_part,
            "identifiers": total_ident,
            "species": total_spe
        }

        total_metrics.append(total_for_month)

    df_monthly = pd.DataFrame(total_metrics)
    # Encontramos el primer índice donde 'observations' > 0 (inicio del proyecto)
    start_index = df_monthly[df_monthly['observations'] > 0].index[0]
    df_final = df_monthly.loc[start_index:].reset_index(drop=True)
    return df_final

In [11]:
df_monthly = build_monthly_metrics(264)

Procesando 2022-01...
Procesando 2022-02...
Procesando 2022-03...
Procesando 2022-04...
Procesando 2022-05...
Procesando 2022-06...
Procesando 2022-07...
Procesando 2022-08...
Procesando 2022-09...
Procesando 2022-10...
Procesando 2022-11...
Procesando 2022-12...
Procesando 2023-01...
Procesando 2023-02...
Procesando 2023-03...
Procesando 2023-04...
Procesando 2023-05...
Procesando 2023-06...
Procesando 2023-07...
Procesando 2023-08...
Procesando 2023-09...
Procesando 2023-10...
Procesando 2023-11...
Procesando 2023-12...
Procesando 2024-01...
Procesando 2024-02...
Procesando 2024-03...
Procesando 2024-04...
Procesando 2024-05...
Procesando 2024-06...
Procesando 2024-07...
Procesando 2024-08...
Procesando 2024-09...
Procesando 2024-10...
Procesando 2024-11...
Procesando 2024-12...
Procesando 2025-01...
Procesando 2025-02...
Procesando 2025-03...
Procesando 2025-04...
Procesando 2025-05...


In [12]:
df_monthly

,month,observations,observers,identifiers,species
0,2022-06,96,4,9,41
1,2022-07,32,5,6,12
2,2022-08,146,6,7,46
3,2022-09,146,8,8,55
4,2022-10,87,3,12,58
5,2022-11,115,2,10,74
6,2022-12,47,3,7,38
7,2023-01,57,4,7,37
8,2023-02,46,2,8,34
9,2023-03,63,5,9,48


In [14]:
observations = get_obs(id_project=264, grade="research")
df_obs, df_photos = get_dfs(observations)

# df_obs.to_csv("data/observations.csv", index=False)
# df_photos.to_csv("data/photos.csv", index=False)

Generating list of observations:
https://api.minka-sdg.org/v1/observations?project_id=264&quality_grade=research&per_page=200
Total observations to download: 4569
Number of elements: 200
Number of elements: 400
Number of elements: 600
Number of elements: 800
Number of elements: 1000
Number of elements: 1200
Number of elements: 1400
Number of elements: 1600
Number of elements: 1800
Number of elements: 2000
Number of elements: 2200
Number of elements: 2400
Number of elements: 2600
Number of elements: 2800
Number of elements: 3000
Number of elements: 3200
Number of elements: 3400
Number of elements: 3600
Number of elements: 3800
Number of elements: 4000
Number of elements: 4200
Number of elements: 4400
Number of elements: 4569


# 3. Taxonomías

Descargamos todas las observaciones del proyecto, usando mecoda_minka. Generamos los dataframes de observaciones y de fotos. A partir de df_obs creamos una función que nos permita ver el número de observaciones por reino, filo, clase... Este rango se tiene que poder indicar como parámetro, para usar la misma función para cualquier rango.

Guardamos df_obs y df_photos como csv en la carpeta `data`. Y no guardamos los índices, como en los casos anteriores.

Creamos la función. Ten en cuenta las columnas de los rangos que tiene el dataframe de df_obs. En las columnas "kingdom", "phylum", "class",... están los rangos taxonómicos superiores a la identificación. El "taxon_name" es el identificado en la observación, y el "taxon_rank" el rango de la identificación. En las otras columnas están los rangos superiores. Esto lo hacemos juntos, que es un poco complicado de explicar por escrito.

In [15]:
def get_taxon_count(df_obs, rank_level):
    """
    Obtiene el conteo de taxones por nivel de rango.
    :param df_obs: DataFrame con las observaciones
    :param rank_level: Nivel de rango (kingdom, phylum, class, order, family, genus, species)
    :return: DataFrame con el conteo de taxones
    """

    if rank_level not in df_obs.columns:
        if rank_level == "species":
            df_taxon_counts = df_obs.loc[df_obs['taxon_rank'] == rank_level, "taxon_name"].value_counts().reset_index()
        else:
            raise ValueError(f"'{rank_level}' esta columna no existe en el DataFrame.")

    if rank_level in df_obs.columns:
        df_taxon_counts = df_obs[rank_level].value_counts().reset_index()
        df_taxon_counts.columns = ["taxon_name", "count"]

    # Si hay observaciones identificadas como este rango
    if rank_level != "species":
        df2 = df_obs.loc[df_obs['taxon_rank'] == rank_level, "taxon_name"].value_counts().reset_index()

        if len(df2) > 0:

            # Concatenamos ambos DataFrames
            df_combined = pd.concat([df_taxon_counts, df2])

            # Agrupamos por taxon_name y sumamos los counts
            df_summed = df_combined.groupby('taxon_name', as_index=False)['count'].sum()
            
            df_sorted = df_summed.sort_values(by='count', ascending=False).reset_index(drop=True)
            
        else:
            df_sorted = df_taxon_counts
    else:
        df_sorted = df_taxon_counts

    df_sorted['taxon_rank'] = rank_level

    return df_sorted[['taxon_rank', 'taxon_name', 'count']]

In [16]:
get_taxon_count(df_obs, "kingdom")

,taxon_rank,taxon_name,count
0,kingdom,Animalia,3117
1,kingdom,Plantae,1330
2,kingdom,Chromista,115
3,kingdom,Fungi,7


In [17]:
get_taxon_count(df_obs, "phylum")

,taxon_rank,taxon_name,count
0,phylum,Tracheophyta,1215
1,phylum,Mollusca,868
2,phylum,Chordata,859
3,phylum,Arthropoda,844
4,phylum,Cnidaria,327
5,phylum,Echinodermata,158
6,phylum,Ochrophyta,112
7,phylum,Rhodophyta,96
8,phylum,Bryozoa,38
9,phylum,Chlorophyta,19


In [18]:
get_taxon_count(df_obs, "class")

,taxon_rank,taxon_name,count
0,class,Magnoliopsida,868
1,class,Aves,744
2,class,Insecta,663
3,class,Bivalvia,519
4,class,Liliopsida,338
5,class,Gastropoda,237
6,class,Echinoidea,140
7,class,Hydrozoa,126
8,class,Scyphozoa,120
9,class,Phaeophyceae,112


# 4. Especies vistas por primera vez en el proyecto desde el último informe (últimos 30 días)

A partir del df_obs podemos sacar este dato fácilmente. Toma el dataframe, ordénalo por fecha de observación, en orden ascendente (las primeras observaciones estarán más arriba). Ahora quédate solo con las primeras observaciones de cada especie. Es decir:
* Seleccionamos aquellas observaciones que hayan llegado al nivel de especie (columna "taxon_rank" == "species").
* Nos quedamos con la primera observación de cada especie, usando drop_duplicates()
```python
df_first = df_obs.drop_duplicates(subset=["taxon_name"], keep="first")
```
* Así nos quedaremos con la primera observación de cada especie. Ahora filtramos de esta tabla las que tengan fecha de observación mayor a hoy menos 30 días (vistas en los últimos 30 días).

Esas serán las especies nuevas observadas en los últimos 30 días.

Primero haz el proceso y luego lo conviertes a una función. Es decir, carga el df_obs y haz los pasos con él, cuando te haya salido ya lo conviertes en función.

In [19]:
def get_new_species(df_obs, last_days=30):
    """
    Obtiene las nuevas especies observadas en los últimos días.
    :param df_obs: DataFrame con las observaciones
    :param last_days: Número de días para considerar una especie como nueva
    :return: DataFrame con las nuevas especies
    """
    df_obs["observed_on"] = pd.to_datetime(df_obs["observed_on"], errors="coerce")

    df_species = df_obs[df_obs["taxon_rank"] == "species"]

    df_species_sorted = df_species.sort_values(by="observed_on")
    df_first = df_species_sorted.drop_duplicates(subset=["taxon_name"], keep="first")

    cutoff_date = datetime.today() - timedelta(days=last_days)
    
    df_new_species = df_first[df_first["observed_on"] > cutoff_date]

    return df_new_species

In [21]:
new_spe = get_new_species(df_obs)
new_spe[["taxon_name", "observed_on", "user_login"]]

,taxon_name,observed_on,user_login
13,Echium plantagineum,2025-04-25,crismc
73,Marcus-kochia littorea,2025-04-28,xasalva
69,Plantago coronopus,2025-04-28,xasalva
76,Austrocylindropuntia cylindrica,2025-04-28,xasalva
66,Olea europaea,2025-04-28,xasalva
79,Aloe vera,2025-04-28,xasalva
95,Exhyalanthrax muscarius,2025-04-28,xasalva
78,Opuntia stricta,2025-04-28,xasalva
132,Daucus pumilus,2025-04-28,xasalva
106,Carpobrotus acinaciformis,2025-04-28,xasalva


Función para sacar una foto de las nuevas especies

In [ ]:
def get_photos_new_species(df_new_species, df_photos):
    # El dataframe df_new_species tiene las especies nuevas, con el id de cada observación
    # Filtramos el dataframe de fotos para quedarnos solo con las fotos de las especies nuevas, las de los ids de esas observaciones.
    # Puedes utilizar el método isin() de pandas para filtrar el dataframe df_photos
    # df_photos['id'].isin(df_new_species['id'])
    # Investiga el método isin() y cómo se utiliza para filtrar un dataframe
    # Nos quedaríamos solo con una foto para cada especie nueva, así que podemos usar el método drop_duplicates() de pandas con subset(['id'])
    
    # Filtrar las fotos cuyas observaciones están en df_new_species
    df_photos_new_species = df_photos[df_photos['id'].isin(df_new_species['id'])]

    # Eliminar duplicados para quedarnos con una foto por observación
    df_photos_new_species = df_photos_new_species.drop_duplicates(subset=['id'])

    return df_photos_new_species

In [22]:
df_new_species = get_new_species(df_obs, last_days=30)
df_new_species

,id,created_at,updated_at,observed_on,observed_on_time,iconic_taxon,taxon_id,taxon_rank,taxon_name,latitude,...,identifiers,num_identification_agreements,num_identification_disagreements,device,kingdom,phylum,class,order,family,genus
13,450036,2025-05-03,2025-05-05,2025-04-25,12:02:00,plantae,245454,species,Echium plantagineum,41.416562,...,"crismc, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Boraginaceae,Echium
73,447664,2025-04-29,2025-05-01,2025-04-28,11:54:00,plantae,264401,species,Marcus-kochia littorea,41.264936,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Capparales,Brassicaceae,Marcus-kochia
69,447668,2025-04-29,2025-05-01,2025-04-28,12:05:00,plantae,242353,species,Plantago coronopus,41.264936,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Plantaginales,Plantaginaceae,Plantago
76,447661,2025-04-29,2025-05-01,2025-04-28,11:52:00,plantae,77959,species,Austrocylindropuntia cylindrica,41.265129,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Caryophyllales,Cactaceae,Austrocylindropuntia
66,447673,2025-04-29,2025-04-29,2025-04-28,12:11:00,plantae,250103,species,Olea europaea,41.265010,...,"xasalva, pauladelmar",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Scrophulariales,Oleaceae,Olea
79,447658,2025-04-29,2025-04-30,2025-04-28,11:50:00,plantae,243911,species,Aloe vera,41.265104,...,"xasalva, bertinhaco",1,0,web,Plantae,Tracheophyta,Liliopsida,Asparagales,Xanthorrhoeaceae,Aloe
95,447639,2025-04-29,2025-05-01,2025-04-28,11:29:00,insecta,264823,species,Exhyalanthrax muscarius,41.264990,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Arthropoda,Insecta,Diptera,Bombyliidae,Exhyalanthrax
78,447659,2025-04-29,2025-04-30,2025-04-28,11:50:00,plantae,250456,species,Opuntia stricta,41.265104,...,"xasalva, bertinhaco",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Caryophyllales,Cactaceae,Opuntia
132,447552,2025-04-29,2025-05-01,2025-04-28,10:39:00,plantae,264824,species,Daucus pumilus,41.264582,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Apiales,Apiaceae,Daucus
106,447590,2025-04-29,2025-04-29,2025-04-28,10:59:00,plantae,249415,species,Carpobrotus acinaciformis,41.265039,...,"xasalva, bertinhaco",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Caryophyllales,Aizoaceae,Carpobrotus


In [23]:
def download_photos(
    df_photos: pd.DataFrame, directorio: str = "minka_photos"
):
    """
    Function to download the photos resulting from the query.
    """
    # Create the folder, if it exists overwrite it
    if not os.path.exists(directorio):
        os.makedirs(directorio)

    session = requests.Session()

    # Iterate through the df_photos query result and download the photos in medium size
    for i, row in df_photos.iterrows():
        response = session.get(row["photos_medium_url"], stream=True)
        if response.status_code == 200:
            with open(f"{directorio}/{row['path']}", "wb") as out_file:
                out_file.write(response.content)
        del response

    # Even using .loc, we get a SettingWithCopyWarning message
    df_photos.loc[:, "abs_path"] = os.path.abspath(f"{directorio}/{df_photos['path']}")


In [25]:
def get_photos_new_species(df_new_species, df_photos):

    # Filtrar las fotos cuyas observaciones están en df_new_species
    df_photos_new_species = df_photos[df_photos['id'].isin(df_new_species['id'])]

    # Eliminar duplicados para quedarnos con una foto por observación
    df_photos_new_species = df_photos_new_species.drop_duplicates(subset=['id'])

    download_photos(df_photos_new_species)

    return df_photos_new_species

In [26]:
get_photos_new_species(df_new_species, df_photos)

,id,photos_id,iconic_taxon,taxon_name,photos_medium_url,user_login,latitude,longitude,license_photo,attribution,path,abs_path
20,450036,606104,plantae,Echium plantagineum,https://minka-sdg.org/attachments/local_photos...,crismc,41.416562,2.231530,cc-by,"(c) crismc, some rights reserved (CC BY)",450036_606104.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
59,447715,602926,plantae,Cutandia maritima,https://minka-sdg.org/attachments/local_photos...,elibonfill,41.264466,1.988138,cc-by-nc,"(c) Elisabet Bonfill i Molina, some rights res...",447715_602926.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
80,447673,602814,plantae,Olea europaea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265010,1.958497,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447673_602814.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
83,447668,602809,plantae,Plantago coronopus,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447668_602809.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
87,447664,602804,plantae,Marcus-kochia littorea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447664_602804.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
91,447661,602802,plantae,Austrocylindropuntia cylindrica,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265129,1.973101,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447661_602802.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
93,447659,602800,plantae,Opuntia stricta,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265104,1.974124,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447659_602800.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
94,447658,602799,plantae,Aloe vera,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265104,1.974124,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447658_602799.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
110,447639,602779,insecta,Exhyalanthrax muscarius,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264990,1.980592,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447639_602779.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
123,447590,602714,plantae,Carpobrotus acinaciformis,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265039,1.985591,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447590_602714.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...


Con esto estaríamos creando las funciones para extraer los datos. Luego estarían las de crear los gráficos y montar el informe.

# 5. Mapa calor para densidad de observaciones

Crear la función que toma un dataframe con el formato de df_obs (con esos nombres de columna, "latitude", "longitude") y lo mapee en el mapa de calor. Ese dataframe puede estar con las observaciones totales, filtrato por un kingdom, por un usuario, por un mes, o por lo que sea, pero no le afecta a la función, que lo hará siempre igual sobre un dataframe con las mismas columnas.

In [ ]:
# Fuente: https://stackoverflow.com/questions/53565979/export-a-folium-map-as-a-png
import io
from PIL import Image

def get_heatmap(df_obs, zoom_start:int):

    df_valid = df_obs.dropna(subset=["latitude", "longitude"])
    heat_data = df_valid[["latitude", "longitude"]].values.tolist()

    mean_lat = df_valid["latitude"].mean()
    mean_lon = df_valid["longitude"].mean()
   

    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=zoom_start)
    HeatMap(heat_data).add_to(m)

    os.makedirs("figures", exist_ok=True)

    html_path = "figures/heatmap.html"
    m.save(html_path)

    img_data = m._to_png(3)
    img = Image.open(io.BytesIO(img_data))
    
    img.save('figures/heatmap_image.png')


In [ ]:
get_heatmap(df_obs, 11)